# Lesson 2 — Document Chunking

**Learning objective:** Understand why we chunk documents before sending them to an LLM, how the deterministic chunker works, and verify the offset invariant that makes span tracking reliable.

---

## Why chunk at all?

Annual reports and press releases are long — often 20,000–100,000 characters. LLMs have context windows and cost constraints. More importantly, a model given an 80-page document produces worse extractions than one given focused 1–2 page passages: it drifts, loses precision, and hallucinates cross-paragraph connections.

**Design choices made here:**
- Deterministic paragraph-aware splitting — no LLM involved in chunking
- Overlap between consecutive chunks to avoid cutting a causal claim at a boundary
- Offset invariant: `text[chunk.start:chunk.end] == chunk.text` — always verifiable

The chunker is ~150 lines of pure Python with no dependencies.

In [ ]:
import sys
sys.path.insert(0, '../../src')

from ptc_extraction.chunking import split_document, Chunk

# Inspect the Chunk dataclass
import dataclasses
print(dataclasses.fields(Chunk))

## Exercise 1 — Split a short document

In [ ]:
# A synthetic Michelin press release fragment
sample_text = """
Michelin Group reported strong third-quarter results despite continued macroeconomic headwinds.
Revenue reached €7.2 billion, up 3.2% year-on-year at constant exchange rates.

The Group faced significant logistics cost pressures stemming from Red Sea disruptions.
Management estimates an additional €45M in freight costs over the first nine months.
These disruptions are not expected to resolve before mid-2025.

Raw material costs, particularly natural rubber, declined 8% versus prior year.
This improvement offset a portion of the logistics headwinds and supported margin recovery.
The Group expects rubber prices to remain broadly stable through year-end.

Currency effects were negative: the stronger euro compressed revenue by approximately €120M in Q3.
Forex headwinds are expected to persist into Q4 given current exchange rate levels.
Specialty segment volumes grew 5.1%, driven by agricultural tyre demand in North America.
""".strip()

chunks = split_document(sample_text, max_chars=400, overlap_chars=80)

print(f"Document length : {len(sample_text)} chars")
print(f"Chunks produced : {len(chunks)}")
print()
for i, c in enumerate(chunks):
    print(f"Chunk {i} | start={c.start:4d} end={c.end:4d} len={len(c.text):3d}")
    print(f"  '{c.text[:80]}...'")
    print()

## The offset invariant

Every chunk satisfies `text[chunk.start:chunk.end] == chunk.text`. This allows downstream code to record the exact span of a PTC inside the original document for audit purposes.

In [ ]:
print("Verifying offset invariant for all chunks:")
all_ok = True
for i, c in enumerate(chunks):
    extracted = sample_text[c.start:c.end]
    ok = (extracted == c.text)
    if not ok:
        all_ok = False
        print(f"  FAIL chunk {i}: '{extracted[:40]}' != '{c.text[:40]}'")

if all_ok:
    print(f"  All {len(chunks)} chunks pass the invariant. ✓")

## Exercise 2 — Overlap ensures continuity

Let's verify that consecutive chunks share text at their boundary, so a causal claim split across a paragraph break still appears in at least one complete chunk.

In [ ]:
if len(chunks) >= 2:
    for i in range(len(chunks) - 1):
        # The start of chunk i+1 should overlap with the end of chunk i
        overlap_text = chunks[i + 1].text[:80]
        present_in_prev = overlap_text[:30] in chunks[i].text
        print(f"Overlap between chunk {i} and chunk {i+1}: {present_in_prev}")
        print(f"  Overlap text: '{overlap_text[:60]}'")
else:
    print("Only one chunk — no overlap needed.")

## Exercise 3 — Size limits hold

With default settings (`max_chars=8000`), no chunk ever exceeds the limit — even for pathologically long single-paragraph documents.

In [ ]:
# Generate a 25,000-character single-paragraph document
import random
import string

rng = random.Random(42)
words = ["rising", "costs", "margins", "revenue", "headwinds", "rubber",
         "logistics", "disruption", "currency", "tariffs", "volume", "growth"]
long_paragraph = " ".join(rng.choices(words, k=4000))

MAX = 8000
big_chunks = split_document(long_paragraph, max_chars=MAX)

print(f"Input length  : {len(long_paragraph):,} chars")
print(f"Chunks        : {len(big_chunks)}")
print(f"Max chunk size: {max(len(c.text) for c in big_chunks):,} chars (limit={MAX})")
assert all(len(c.text) <= MAX for c in big_chunks), "FAIL: a chunk exceeded max_chars!"
print("All chunks within limit. ✓")

## What gets passed to the LLM

Each chunk is combined with document metadata to form the final user prompt. The source_date and source_type fields from the original document are injected so the LLM can apply the direction rule correctly.

In [ ]:
from ptc_extraction.prompts import USER_PROMPT_TEMPLATE_V1

example_prompt = USER_PROMPT_TEMPLATE_V1.format(
    source_date="2024-11-14",
    source_type="management",
    chunk_text=chunks[0].text,
)

print(example_prompt[:600])
print("...")

---

## Key takeaways

1. Chunk documents **before** sending to LLM — precision improves dramatically.
2. The offset invariant (`text[start:end] == chunk.text`) enables exact span tracking for every extracted PTC.
3. Overlap prevents causal claims at paragraph boundaries from being lost.
4. The chunker is deterministic and dependency-free — no LangChain, no sentence-transformers.

**Next:** Lesson 3 — Extraction: from chunks to PTCs via structured LLM calls.